# Minimizing a <span style="font-variant:small-caps;">Fsm</span>

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "AutomataTypes.fsx"
#load "NfaToDfa.fsx"
#load "Minimize.fsx"

In [ ]:
open AutomataTypes
open NfaToDfa
open Minimize

This version factors the two finite, monotone fixed-point computations into reusable functions. The original notebook refers to `FixedPoint.ipynb`, which is absent from the repository. The two required functions are fully defined here, so this notebook has no missing dependency.

`fixpoint` expands each element separately. `fixpoint2` expands the entire current set. Both retain every previously found element and stop when the finite set no longer changes.

In [ ]:
let fixpoint seed successors =
    let rec expand reached =
        let next = reached |> Seq.map successors |> Set.unionMany |> Set.union reached
        if next = reached then reached else expand next
    expand seed

let fixpoint2 seed expandSet =
    let rec expand reached =
        let next = Set.union reached (expandSet reached)
        if next = reached then reached else expand next
    expand seed

The function `cart_prod(A, B)` computes the *Cartesian product* $A \times B$ of the sets $A$ and $B$ where
$$ A \times B := \{ (x, y) \mid x \in A \wedge y \in B \}. $$

`cartesianProduct`, `separate` and `quotient` are imported from the preceding minimization notebook. This retains one owner for equivalence classes and the mathematical pair-separation rule.

This function takes three arguments:
- `Pairs`  a set `Pairs` of pairs of states from some <span style="font-variant:small-caps;">Fsm</span> $F$.
   If $(p_1, p_2) \in \texttt{Pairs}$, then $p_1$ and $p_2$ are known to be *separable*,
- `States` is the set of states of the <span style="font-variant:small-caps;">Fsm</span> $F$,
- `Sigma`  is the alphabet of the underlying <span style="font-variant:small-caps;">Fsm</span> $F$.

The function `separate(Pairs)` computes the set of pairs of states $(q_1, q_2)$ that are separable because there is some
$c \in \Sigma$ such that 
$$\delta(q_1,c) = p_1, \quad \delta(q_2,c) = p_2. $$

The function `minimize(A)` takes a deterministic 
<span style="font-variant:small-caps;">Fsm</span> `F` as its input.
The five record fields correspond to
$$ F = (Q, \Sigma, \delta, q_0, A) $$
The algorithm performs the following steps:
1. All unreachable states are eliminated.
2. All accepting states are separated form all non-accepting states.
3. States are separated as long as possible.
   Two states $p_1$ and $p_2$ are separable if there is a character 
   $c \in \Sigma$ such that 
   $$\delta(p_1,c) = q_1, \quad \delta(p_2,c) = q_2, \quad \textrm{and} \quad
     \mbox{$q_1$ and $q_2$ are separable.}
   $$
4. States that are not separable are *equivalent* and are therefore identified and grouped
   in equivalence classes.

In [ ]:
let minimizeFP (dfa: Dfa<'state>) : Dfa<Set<'state>> =
    requireComplete dfa
    let states = fixpoint (Set.singleton dfa.Start) (fun q -> dfa.Alphabet |> Set.map (fun c -> dfa.Transitions[q, c]))
    let accepting = Set.intersect states dfa.Accepting
    let nonAccepting = Set.difference states accepting
    let initial = Set.union (cartesianProduct nonAccepting accepting) (cartesianProduct accepting nonAccepting)
    let separable = fixpoint2 initial (fun pairs -> separate pairs states dfa.Alphabet dfa.Transitions)
    quotient dfa states separable

Both minimization implementations produce the same partition and transition map. This example also includes an unreachable accepting state.

In [ ]:
let example : Dfa<int> =
    { States = set [0;1;2;3]; Alphabet = set ['a']
      Transitions = Map.ofList [ (0,'a'),1; (1,'a'),2; (2,'a'),1; (3,'a'),3 ]
      Start = 0; Accepting = set [1;2;3] }
checkNotebook (minimizeFP example = minimize example)
checkNotebook (fixpoint (set [0]) (fun n -> if n < 3 then set [n+1] else Set.empty) = set [0;1;2;3])
minimizeFP example